# 60 — Protocollo del relatore sul dataset per file

Esecuzione del notebook-protocollo v0.1 del relatore sui **file originali** del dataset (cartella `monaco/rf_jamming_dataset` sul server): 96.090 registrazioni, active e passive scan.

**Modifiche rispetto al notebook del relatore**, tutte limitate al caricamento e alle risorse; modelli, metriche, soglie e gate sono invariati:
1. la copia sul server non ha il `manifest.csv` della release 2026, quindi il manifest viene **ricostruito dai percorsi** con `qdetect.files.build_manifest`: etichetta, modalità, sottotipo benigno, forma d'onda, potenza, canale, banda e luogo. Viene salvato tra i risultati;
2. `DATA_ROOT` punta alla cartella del server e i risultati vanno in `results/protocollo_file/` della repo;
3. `n_jobs` è limitato (`N_JOBS`, default 16), perché il server è condiviso;
4. `CAP_PER_STRATUM` è configurabile da variabile d'ambiente, per una prima esecuzione ridotta.

Come nell'originale: un prefisso per file, split per hash del contenuto del file (50/25/25), training sui soli benigni, soglia conforme sul fondo in camera, nessun vantaggio quantistico assunto.

In [ ]:
# Uncomment if needed:
# %pip install kagglehub numpy pandas scipy scikit-learn matplotlib nbformat
import os, json, hashlib, time, itertools, pathlib, platform, shutil
import numpy as np, pandas as pd, scipy, sklearn, matplotlib.pyplot as plt
from scipy.stats import beta
from scipy.linalg import eigh
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.kernel_approximation import RBFSampler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics.pairwise import rbf_kernel, polynomial_kernel, linear_kernel
from IPython.display import display
SEED=42; rng=np.random.default_rng(SEED)
SMOKE=os.environ.get('SMOKE','0')=='1'
import sys
REPO=pathlib.Path.cwd()
while not (REPO/'qdetect').exists() and REPO!=REPO.parent: REPO=REPO.parent
sys.path.insert(0,str(REPO))
from qdetect.files import build_manifest
DATA_ROOT=os.environ.get('QS_DATA_ROOT','/home/habes/NotebookProjects/bboi/monaco/rf_jamming_dataset')
N_JOBS=int(os.environ.get('N_JOBS','16'))
OUT=REPO/'results'/'protocollo_file'/os.environ.get('RUN_NAME','full'); OUT.mkdir(parents=True,exist_ok=True)
BUDGETS=[8,16,32,64,128]; ALPHAS=[.01,.001,.0001]; CAP_PER_STRATUM=int(os.environ.get('CAP_PER_STRATUM','1200'))
FIELDS=['noise','max_magnitude','total_gain_db','base_pwr_db','rssi','relpwr_db','avgpwr_db']
CORE=[x for x in FIELDS if x not in ['noise','total_gain_db']]
CFG=dict(seed=SEED,smoke=SMOKE,budgets=BUDGETS,alphas=ALPHAS,cap_per_stratum=CAP_PER_STRATUM,fields=FIELDS,core=CORE,split='sha256_file_content_50_25_25',risk_unit='file_prefix',q_qubits=6,q_sample=96)
(OUT/'design.json').write_text(json.dumps(CFG,indent=2)); print(CFG)


## E1 — metadata and provenance
Current release: `release_artifacts/metadata/manifest.csv`, raw `rf_jamming/`. Require documented manifest instead of guessing labels from path. A legacy release without manifest stops; provide a verified manifest with these fields. File content hashes prevent exact duplicates from crossing splits. The manifest does not establish independent sessions.

In [ ]:
if SMOKE:
    root=OUT/'synthetic'; root.mkdir(exist_ok=True)
    rows=[]
    for mode in ['active','passive']:
        for label,sub,w,p in [('benign','floor','',np.nan),('benign','background','',np.nan),('malicious','','gaussian_noise',-40),('malicious','','single_tone',0)]:
            for i in range(64):
                f=root/f'{mode}_{label}_{sub}_{w}_{i}.csv'
                a=rng.normal(size=(128,len(FIELDS)))
                if label=='malicious': a[:,1]+=.7; a[:,3]+=a[:,0]*a[:,1]
                d=pd.DataFrame(a,columns=FIELDS); d['freq1']=5805; d.to_csv(f,index=False)
                rows.append(dict(file_path=f.name,scan_mode=mode,label=label,benign_subtype=sub,waveform=w,power_dbm=p,channel_mhz=5805,location='synthetic'))
    manifest=pd.DataFrame(rows)
else:
    root=pathlib.Path(DATA_ROOT).resolve()
    candidates=[x for x in root.rglob('manifest.csv') if x.parent.name=='metadata'] if (root/'release_artifacts').exists() else []
    if len(candidates)==1:
        mf=candidates[0]; manifest=pd.read_csv(mf); print('Manifest:',mf)
    else:
        # Nessun manifest documentato: lo si ricostruisce dai percorsi (qdetect.files).
        manifest=build_manifest(root); manifest.to_csv(OUT/'manifest_from_paths.csv',index=False)
        print('Manifest ricostruito dai percorsi:',len(manifest),'file')
        print(manifest.groupby(['scan_mode','label','benign_subtype','waveform']).size())
required=['file_path','scan_mode','label','benign_subtype','waveform','power_dbm','channel_mhz']
assert set(required)<=set(manifest), f'STOP missing {set(required)-set(manifest)}'
for c in ['label','scan_mode','benign_subtype','waveform']:
    manifest[c]=manifest[c].fillna('').astype(str).str.strip().str.lower()
manifest['scan_mode']=manifest.scan_mode.str.replace('_scan','',regex=False)
assert set(manifest.label)=={'benign','malicious'}, manifest.label.unique()
assert not manifest.file_path.duplicated().any(), 'Duplicate manifest paths'
strata=['scan_mode','label','benign_subtype','waveform','power_dbm','channel_mhz']
manifest.groupby(strata,dropna=False).size().rename('files').to_csv(OUT/'inventory.csv')
parts=[]
for _,g in manifest.groupby(strata,dropna=False,sort=True):
    parts.append(g.sample(min(len(g),CAP_PER_STRATUM),random_state=SEED))
selected=pd.concat(parts).sort_values('file_path').reset_index(drop=True)
print('Corpus',len(manifest),'sample',len(selected)); display(selected.head())


In [ ]:
def resolve_path(s):
    f=(root/str(s)).resolve()
    if not f.is_file(): f=(root/'rf_jamming'/str(s)).resolve()
    if not f.is_relative_to(root.resolve()) or not f.is_file(): raise FileNotFoundError(str(s))
    return f
records=[]; arrays=[]; failures=[]; seen=set()
for r in selected.to_dict('records'):
    try:
        f=resolve_path(r['file_path']); h=hashlib.sha256(f.read_bytes()).hexdigest()
        if h in seen: raise ValueError('duplicate_content')
        d=pd.read_csv(f,nrows=max(BUDGETS)); a=d[FIELDS].apply(pd.to_numeric,errors='coerce').to_numpy(float)
        if len(a)!=max(BUDGETS) or not np.isfinite(a).all(): raise ValueError('short_or_nonfinite_prefix')
        seen.add(h); u=int(h[:12],16)/16**12
        records.append(dict(r,sha256=h,split='train' if u<.5 else 'cal' if u<.75 else 'test',freq_count=d['freq1'].nunique() if 'freq1' in d else np.nan))
        arrays.append(a)
    except Exception as e: failures.append(dict(path=r['file_path'],reason=str(e)))
pd.DataFrame(failures,columns=['path','reason']).to_csv(OUT/'excluded.csv',index=False)
M=pd.DataFrame(records); A=np.asarray(arrays)
if not len(M): raise RuntimeError('STOP no valid raw files; inspect excluded.csv')
M.to_csv(OUT/'sample_manifest.csv',index=False); assert M.sha256.is_unique
print(M.groupby(['scan_mode','split','label']).size()); print('excluded',len(failures))


## E2–E4 — equal-risk, feature attribution, observation budgets
Prefix is the decision unit; same files at every budget. Fit raw scaling only on benign training rows. Representations: first-order mean; second-order mean/covariance; higher order adds marginal skew/kurtosis and mixed third moments; temporal adds lag-1 cross-products. Shuffled control preserves all within-window simultaneous moments while removing order. Core excludes noise and gain. No metadata features. Nominal power is not JNR. File-level confidence bounds assume independent trials and are descriptive only. Cross-mode thresholds are calibrated on source only.

In [ ]:
def features(a,kind):
    mu=a.mean(1); z=a-mu[:,None,:]; L,p=a.shape[1:]; cov=np.einsum('nli,nlj->nij',z,z)/L
    i,j=np.triu_indices(p); out=[mu]
    if kind!='mean': out.append(cov[:,i,j])
    if kind in ['higher','temporal','shuffled']:
        out.extend([(z**3).mean(1),(z**4).mean(1)])
        out.append(np.stack([(z[:,:,i]*z[:,:,j]*z[:,:,k]).mean(1) for i,j,k in itertools.combinations_with_replacement(range(p),3)],axis=1))
    if kind in ['temporal','shuffled']:
        if kind=='shuffled':
            rr=np.random.default_rng(SEED); z=np.stack([x[rr.permutation(L)] for x in z])
        out.append((np.einsum('nli,nlj->nij',z[:,:-1],z[:,1:])/(L-1)).reshape(len(a),-1))
    return np.concatenate(out,axis=1)
def threshold(cal,alpha):
    k=int(np.ceil((len(cal)+1)*(1-alpha)))
    return np.sort(cal)[k-1] if 0<k<=len(cal) else np.inf
def upper(k,n): return float(beta.ppf(.95,k+1,n-k)) if k<n and n else (1. if n else np.nan)
def models(X,train,yt):
    B=X[train & (yt==0)]
    lw=LedoitWolf().fit(B); yield 'mahalanobis',lw.mahalanobis
    pca=PCA(n_components=max(1,min(B.shape[1]-1,len(B)-1,8)),random_state=SEED).fit(B)
    yield 'pca_residual',lambda x: ((x-pca.inverse_transform(pca.transform(x)))**2).sum(1)
    iso=IsolationForest(n_estimators=80,random_state=SEED,n_jobs=N_JOBS).fit(B); yield 'isolation_forest',lambda x:-iso.score_samples(x)
    # cap kernel training size deterministically to bound computation
    bb=B[np.random.default_rng(SEED).choice(len(B),min(1500,len(B)),replace=False)]
    oc=OneClassSVM(gamma='scale',nu=.05).fit(bb); yield 'ocsvm',lambda x:-oc.decision_function(x).ravel()
    rff=RBFSampler(gamma=1/max(1,B.shape[1]),n_components=128,random_state=SEED).fit(B)
    cov=LedoitWolf().fit(rff.transform(B)); yield 'rff_mahalanobis',lambda x:cov.mahalanobis(rff.transform(x))
    if len(np.unique(yt[train]))==2:
        rf=RandomForestClassifier(n_estimators=80,min_samples_leaf=3,n_jobs=N_JOBS,random_state=SEED,class_weight='balanced').fit(X[train],yt[train]); yield 'supervised_rf',lambda x:rf.predict_proba(x)[:,1]
Y=(M.label=='malicious').to_numpy(int)
res=[]; timing=[]; skips=[]
def evaluate(scores,cal,test,tag):
    neg=test & (Y==0); pos=test & (Y==1)
    if not neg.any() or not pos.any() or not cal.any(): return
    auc=roc_auc_score(Y[test],scores[test]); ap=average_precision_score(Y[test],scores[test])
    groups=[('all',pos)]
    for cols in [['power_dbm'],['waveform'],['channel_mhz'],['power_dbm','waveform','channel_mhz']]:
        for val,g in M[pos].groupby(cols,dropna=False):
            mask=np.zeros(len(M),bool); mask[g.index]=True; groups.append(('/'.join(cols)+':'+str(val),mask))
    for alpha in ALPHAS:
        h=threshold(scores[cal],alpha); pred=scores>h; fp=int((pred&neg).sum()); nn=int(neg.sum())
        for name,mask in groups:
            res.append(dict(tag,stratum=name,alpha=alpha,threshold=h,finite_threshold=bool(np.isfinite(h)),n_cal=int(cal.sum()),n_neg=nn,n_pos=int(mask.sum()),fpr=fp/nn,fpr_upper95_file_iid=upper(fp,nn),tpr=float(pred[mask].mean()),auroc=roc_auc_score(Y[neg|mask],scores[neg|mask]),ap=average_precision_score(Y[neg|mask],scores[neg|mask])))


In [ ]:
# Same-mode floor matched, background OOD; cross-mode diagnostic at budget 128.
modes=sorted(M.scan_mode.unique()); specs=[]
for s in modes:
    specs.append((s,s,'floor'))
    specs.append((s,s,'background'))
    for t in modes:
        if s!=t: specs.append((s,t,'floor'))
for source,target,sub in specs:
    src=(M.scan_mode==source).to_numpy(); dst=(M.scan_mode==target).to_numpy(); floor=(M.benign_subtype=='floor').to_numpy(); targetben=(M.benign_subtype==sub).to_numpy()
    tr=src & (M.split=='train').to_numpy() & ((Y==1)|floor)
    ca=src & (M.split=='cal').to_numpy() & (Y==0) & floor
    te=dst & (M.split=='test').to_numpy() & ((Y==1)|targetben)
    if (tr&(Y==0)).sum()<10 or ca.sum()<5 or len(np.unique(Y[te]))<2:
        skips.append(dict(source=source,target=target,subtype=sub,reason='insufficient_data')); continue
    budgets=BUDGETS if source==target and sub=='floor' else [128]
    for b in budgets:
        for fields in ['core','all'] if b==128 else ['core']:
            ii=[FIELDS.index(x) for x in (CORE if fields=='core' else FIELDS)]
            a=A[:,:b,ii]; sc=StandardScaler().fit(a[tr&(Y==0)].reshape(-1,len(ii))); a=np.clip(sc.transform(a.reshape(-1,len(ii))).reshape(a.shape),-10,10)
            kinds=['mean','second','higher','temporal','shuffled'] if source==target and sub=='floor' else ['mean','temporal']
            for kind in kinds:
                f=features(a,kind); fs=StandardScaler().fit(f[tr&(Y==0)]); X=fs.transform(f)
                start=time.perf_counter()
                for name,predict in models(X,tr,Y):
                    ts=time.perf_counter(); scores=predict(X); elapsed=time.perf_counter()-ts
                    tag=dict(source=source,target=target,benign_subtype=sub,budget=b,fields=fields,representation=kind,model=name,training='supervised' if name=='supervised_rf' else 'benign_only')
                    evaluate(scores,ca,te,tag); timing.append(dict(tag,score_seconds=elapsed))
                timing.append(dict(source=source,target=target,benign_subtype=sub,budget=b,fields=fields,representation=kind,model='all_models_fit_score',score_seconds=time.perf_counter()-start))
    pd.DataFrame(res).to_csv(OUT/'metrics.csv',index=False)
pd.DataFrame(timing).to_csv(OUT/'timing.csv',index=False); pd.DataFrame(skips).to_csv(OUT/'skipped_regimes.csv',index=False)
R=pd.DataFrame(res)
if R.empty: raise RuntimeError('STOP: no eligible strata')
print('metrics',len(R)); display(R.head())


In [ ]:
# Descriptive best observed envelope; not a model selected for deployment.
V=R[(R.stratum=='all') & (R.alpha==.01)&(R.training=='benign_only')&(R.fields=='core')&(R.source==R.target)&(R.benign_subtype=='floor')&R.finite_threshold]
for mode,g in V.groupby('source'):
    fig,ax=plt.subplots(figsize=(8,4))
    for kind,h in g.groupby('representation'):
        z=h.groupby('budget').tpr.max(); ax.plot(z.index,z.values,marker='o',label=kind)
    ax.set(xlabel='Reports per file',ylabel='Exploratory maximum TPR',ylim=(0,1.05),title=mode+' — nominal FPR 1%'); ax.legend(); fig.tight_layout(); fig.savefig(OUT/f'budget_{mode}.png',dpi=150); plt.show()
V.to_csv(OUT/'matched_low_fpr.csv',index=False)
# paired model/representation differences on the same test set; no causal claim
keys=['source','target','benign_subtype','budget','fields','model','training','stratum','alpha']
piv=R[R.finite_threshold].pivot_table(index=keys,columns='representation',values='tpr',aggfunc='first').reset_index()
for x,y in [('higher','second'),('temporal','higher'),('temporal','shuffled')]:
    if x in piv and y in piv: piv[x+'_minus_'+y]=piv[x]-piv[y]
piv.to_csv(OUT/'attribution.csv',index=False)


## E5 — sequential eligibility
The documented schema has ordered reports, not validated physical timestamps or attack onsets. A sequential detection experiment needs continuous benign-to-jamming recordings, verified onset, clock and independent benign runs. Do not infer time or concatenate files. This version explicitly stops E5.

In [ ]:
(OUT/'sequential_gate.json').write_text(json.dumps(dict(status='STOP_ONSET_AND_CLOCK_NOT_VERIFIED',required=['continuous capture','verified onset','timestamp units and clock','independent benign sequences'],future_metrics=['Lorden delay','ARL under H0','missed events']),indent=2))


## E6 — geometric prescreen (training-only, exact statevector)
A 6-qubit feature map: RY input, ring CNOT, data re-uploading RY. Separable control uses identical rotations without CNOT. Common benign-fitted PCA and scaling for all kernels. This evaluates a finite sample geometry, not learning advantage. No labels optimized, no quantum timing claim. Ridge sensitivity and trace normalization are explicit.

In [ ]:
def state(x,ent=True):
    q=len(x); v=np.zeros(2**q); v[0]=1.; idx=np.arange(len(v))
    def ry(v,j,t):
        i=idx[(idx&(1<<j))==0]; k=i|(1<<j); out=v.copy(); c,s=np.cos(t/2),np.sin(t/2); out[i]=c*v[i]-s*v[k]; out[k]=s*v[i]+c*v[k]; return out
    for j,t in enumerate(x): v=ry(v,j,t)
    if ent:
        for j in range(q):
            k=(j+1)%q; perm=idx^(((idx>>j)&1)<<k); v=v[perm]
    for j,t in enumerate(x): v=ry(v,j,t*.7)
    return v
qrows=[]; q=6
for mode in modes:
    tr=(M.scan_mode==mode).to_numpy()&(M.split=='train').to_numpy(); ben=tr&(Y==0)&(M.benign_subtype=='floor').to_numpy()
    if ben.sum()<q+1 or tr.sum()<20: continue
    a=A[:,:128,[FIELDS.index(x) for x in CORE]]; a=a.reshape(len(A),-1)
    sc=StandardScaler().fit(a[ben]); z=sc.transform(a); pp=PCA(n_components=q,random_state=SEED).fit(z[ben]); z=pp.transform(z); zs=StandardScaler().fit(z[ben]); z=np.pi*np.tanh(zs.transform(z)/2)
    ids=np.random.default_rng(SEED).choice(np.flatnonzero(tr),min(96,tr.sum()),replace=False); x=z[ids]; N=len(x)
    Ks={}
    for ent in [False,True]:
        v=np.stack([state(w,ent) for w in x]); assert np.allclose((v*v).sum(1),1); Ks['entangled' if ent else 'separable']=(v@v.T)**2
    controls={'linear':linear_kernel(x),'separable':Ks['separable']}
    for gamma in [.1,1/q,1.]: controls[f'rbf_{gamma}']=rbf_kernel(x,gamma=gamma)
    for d in [2,3,4]: controls[f'poly_{d}']=polynomial_kernel(x,degree=d,gamma=1/q,coef0=1)
    Kq=Ks['entangled']; off=Kq[np.triu_indices(N,1)]; ev=np.linalg.eigvalsh(Kq); rank=float(ev.sum()**2/(ev@ev))
    for name,K in controls.items():
        K=K*N/np.trace(K)
        for ridge in [1e-6,1e-4,1e-2,.1]:
            # symmetric ridge on both matrices, renormalized traces
            g=np.sqrt(max(0,eigh(Kq+ridge*np.eye(N),K+ridge*np.eye(N),eigvals_only=True)[-1]))
            qrows.append(dict(mode=mode,control=name,ridge=ridge,g=float(g),N=N,q=q,effective_rank=rank,offdiag_mean=off.mean(),offdiag_std=off.std(),pairs=N*(N-1)//2))
    pd.DataFrame(Kq).to_csv(OUT/f'quantum_gram_{mode}.csv',index=False)
pd.DataFrame(qrows).to_csv(OUT/'geometry.csv',index=False); display(pd.DataFrame(qrows).head())


## E7 — resource accounting and conservative gate
Hoeffding shot upper-budget proxy for each pair (not lower bound); no QPU measurements. A simultaneous guarantee uses a union bound. Finite threshold and descriptive upper FPR must both be inspected. Low performance is not a proof of information-theoretic hardness.

In [ ]:
cost=[]
for N in [96,256,1024]:
    pairs=N*(N-1)//2
    for eps in [.05,.02,.01]:
        shots=int(np.ceil(np.log(2*pairs/.05)/(2*eps**2)))
        cost.append(dict(N=N,pairs=pairs,epsilon=eps,family_failure=.05,shots_per_pair=shots,total_shots=pairs*shots,encoding_ry_per_state=2*q,cnot_per_state=q))
pd.DataFrame(cost).to_csv(OUT/'cost_proxy.csv',index=False)
# Candidate regimes using an observed envelope, not certification. No automatic quantum GO.
W=R[(R.alpha==.01)&R.finite_threshold&(R.fields=='core')&(R.training=='benign_only')&(R.source==R.target)&(R.benign_subtype=='floor')]
cols=['source','budget','stratum']; rows=[]
for key,g in W.groupby(cols):
    feasible=g[g.fpr_upper95_file_iid<=.01]
    status='INSUFFICIENT_RISK_EVIDENCE' if feasible.empty else ('CLASSICAL_SATURATION_CANDIDATE' if feasible.tpr.max()>=.95 else 'DIFFICULTY_CANDIDATE_NOT_CERTIFIED')
    rows.append(dict(zip(cols,key),status=status,best_tpr=float(feasible.tpr.max()) if len(feasible) else None))
pd.DataFrame(rows).to_csv(OUT/'difficulty_candidates.csv',index=False)
summary=dict(revision='protocollo-relatore-v0.1-file',mode='SYNTHETIC_SMOKE_ONLY' if SMOKE else 'REAL_EXPLORATORY_ONLY',files=len(M),excluded=len(failures),metric_rows=len(R),geometry_rows=len(qrows),independent_sessions_verified=False,sequential_ready=False,quantum_advantage_demonstrated=False,decision='STOP_CONFIRMATORY__INDEPENDENT_SESSIONS_AND_RISK_REQUIRED',design_sha256=hashlib.sha256((OUT/'design.json').read_bytes()).hexdigest(),versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__,sklearn=sklearn.__version__))
(OUT/'summary.json').write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2)); print(shutil.make_archive(str(OUT),'zip',OUT))
